## Build the Shor's Algorithm in QpiAI Explorer Lab on Amazon Braket ##

In this notebook, we implement the Shor's algorithm using the Amazon Braket SDK and run a simple example of factoring the number 15. The students are encouraged to check how it works with other numbers.

 Shor's algorithm is used to find prime factors of an integer. On a quantum computer, Shor's algorithm runs in polynomial time and is almost exponentially faster than the most efficient known classical factoring algorithm. The efficiency of Shor's algorithm is due to the efficiency of the Quantum Fourier transform, Quantum Phase estimation and modular exponentiation by repeated squarings.

### Structure of the algorithm

Shor's Algorithm consists of two parts:

1. Classical part : Reduction of the factoring problem into an order finding problem, which can be done on a classical computer.

2. Quantum part: A Quantum algorithm to solve the order-finding problem.


Shor's algorithm hinges on a result from number theory, which says:

The function $f(a) = x^a$ mod $n$ is a periodic function, where $x$ is an integer coprime to $n$. In the context of Shor's algorithm $n$ will be the number we wish to factor. When two numbers are coprime it means that their greatest common divisor is 1.

The reason why this function is of utility in factoring large numbers is that since $f(a)$ is a periodic function, it has some period $r$. We know that $x^0\mod n = 1$, so $x^r\mod n = 1$ and $x^{2r}\mod n = 1$, and so on since the function is periodic.

\begin{eqnarray}
x^r \equiv 1\mod nn  \\
(x^{\frac{r}{2}})^2 \equiv x^r \equiv 1\mod n  \\
(x^{\frac{r}{2}})^2 - 1 \equiv 0\mod n
\end{eqnarray}


And if $r$ is an even number:

\begin{equation} (x^{\frac{r}{2}} -1)(x^{\frac{r}{2}} +1) \equiv 0\mod n \end{equation}

This means that the product $(x^{\frac{r}{2}} -1)(x^{\frac{r}{2}} +1) $ is an integer multiple of $n$, the number to be factored. So long as $x^{\frac{r}{2}}$ is not equal to $\pm 1$ , then at least one of $(x^{\frac{r}{2}} -1)$ or $(x^{\frac{r}{2}} +1)$ must have a nontrivial factor in common with $n$. So by computing $gcd(x^{\frac{r}{2}}-1, n)$ and $gcd(x^{\frac{r}{2}}+1, n)$, we will obtain a factor of $n$, where $gcd$ is the greatest common divisor function.





### Phases:

The Shor's algorithm requires two quantum registers. At the beginning of the algorithm, one has to choose $q = 2^s$ for some integer $s$ such that $n^2 ≤ q < 2n^2$, where $n$ is to be factored. More precisely, it proceeds as follows:

1. Create a quantum register and partition it into two parts, register 1 and register 2. Thus the state of our quantum computer is given by: $|reg1, reg2\rangle$.

Register 1 must have enough qubits to represent the integers as large as $q - 1$, while register 2 must have enough qubits to represent integers as large as $n - 1$ (the calculation for how many qubits are needed would be done on a classical computer).

2. Initialization: put register 1 in the uniform superposition, i.e. an equally weighted superposition of all integers from $0$ to $q - 1$ (a superposition of the all the rows of the table shown above), and load register 2 with all zeros.

$$ \frac{1}{\sqrt{q}}\sum_{a=0}^{q-1}|a\rangle|0\rangle$$

3. Now apply the transformation $x^a \mod n$ to for each number stored in register 1 and store the result in register 2. The new state is:

$$ \frac{1}{\sqrt{q}}\sum_{a=0}^{q-1}|a\rangle|x^a \mod n\rangle$$

### Modular Exponentiation

The bottleneck in the quantum factoring algorithm is modular exponentiation. The modular exponentiation problem, given $n$, $x$ and $r$, find $x^r \mod n$. Fundamentally, quantum modular exponentiation is $O(n^3)$; this means that the number of quantum gates or operations scales with the cube of the length in bits of the number to be factored. It consists in $2n$ modular multiplications, each of which consists of $O(n)$ additions, each of which requires $O(n)$ operations. However, $O(n^3)$ operations do not necessarily require $O(n^3)$ time steps.

On an abstract machine, it is relatively straightforward to see how to reduce each of those three layers to time steps, in exchange for more space and more total gates, giving a total running time of $O(\log{n}^3)$ if $O(n^3)$ qubits are available and an arbitrary number of gates can be executed concurrently on separate qubits. Such large numbers of qubits are not expected to be practical for the foreseeable future, that is why some people are trying to optimize this process.

1. Measure the second register, and observe some value $k$. This has the side effect of collapsing register one into a equal superposition of each value $a$ between $0$ and $q - 1$ such that $x^a \mod n = k$. The new state is:

$$ \frac{1}{\sqrt{||A||}} \sum_{a^{\prime} \in A} |a^{\prime\rangle, k}$$

where $A$ is the set of $a^{\prime}$ s such that $x^a \mod n = k$ and $||A||$ is the number of elements in that set.

2. Compute the quantum Fourier transform on register one. The aim of applying QFT to the register 1 is to accumulate the wanted state $|a, x^a \mod n\rangle$ such that it is possibly observed with significant probability.

The QFT, when applied to a state $|a\rangle$, changes it in the following manner:

$$ |a\rangle = \frac{1}{\sqrt{q}}\sum_{c=0}^{q-1} |c\rangle e^{\frac{2\pi iac}{q}}$$

So the new state is:

$$ \frac{1}{\sqrt{||A||}}\sum_{a^{\prime} \in A} \frac{1}{\sqrt{q}} \sum_{c=0}^{q-1} |c, k\rangle e^{\frac{2\pi i a^{\prime}c}{q}} $$

3. Measure the state of register one, call this value m, this integer m has a very high probability of being a multiple of $ \frac{q}{r} $, where r is the desired period.

4. At this point we repeat the algorithm to retrieve several distinct multiples of $\frac{q}{r}$. Once we have enough values, we can compute their GCD (with Euclid's algorithm) to retrieve $\frac{q}{r}$. $q$ is given by the problem, so it is easy to compute $r$.

**NOTE: The last point is true under one important assumption, i.e. that $r$ divides $q$.**

Let's assume that $\frac{q}{r} > 2r$, meaning that the number of periods that we look at is comparable to and larger than the period itself. Actually, the quantum circuit remains the same but let's suppose that at the end we measure a value $L$.
The claim is that:

$$ \frac{L}{q} \sim \frac{t}{r}$$
for some integer $t$.
$L$ is the output of the algorithm, $q$ is known, $r$ is the period we want to retrieve and $t$ is unknown.

So how do we find both $t$ and $r$? It turns out that $\frac{t}{r}$ is the best approximation to $\frac{L}{q}$ with a denominator as small as $r$, so we use the fact that $r$ is much smaller than $\sqrt{q}$.
Given that you can reconstruct $\frac{t}{r}$ using a technique called continued fractions, which can be performed very quickly on a classical computer.

### The Braket implemetation of the algorithm starts here:

In [1]:
! pip install amazon-braket-sdk
! pip install RISE

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.7/313.7 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.6/125.6 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.0/224.0 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 191.7/191.7 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.2/139.2 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 377.0/377.0 kB 8.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 539.8/539.8 kB 8.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 kB 10.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 22.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.7/82.7 kB 9.1 MB/s eta 0:00:00
  Created wheel for antlr4-python3-runtime: filename=antlr4_python3_runtime-4.9.2-py3-none-any.whl size=144548 sha256=81a9a1701fa3f77510bdfba

In [2]:
# install required packages and import relevant functions

import math
from collections import Counter
from fractions import Fraction
from typing import Any, Dict, List, Optional

import numpy as np
from braket.circuits import Circuit, circuit
from braket.circuits.qubit_set import QubitSetInput
from braket.devices import Device
from braket.devices import LocalSimulator

 ### Function definitions begin here for building up the ultimate 'run_shors_algorithm()' method. Kindly follow the code cells from this function and map it up to a tree of methods used sequentially.

In [3]:
def inverse_qft_noswaps(qubits: QubitSetInput) -> Circuit:
    """
    Construct a circuit object corresponding to the inverse Quantum Fourier Transform (QFT)
    algorithm, applied to the argument qubits.  Does not use recursion to generate the circuit.

    Args:
        qubits (QubitSetInput): Qubits on which to apply the inverse Quantum Fourier Transform

    Returns:
        Circuit: Circuit object that implements the inverse Quantum Fourier Transform algorithm
    """
    # Instantiate circuit object
    qft_circuit = Circuit()

    # Fet number of qubits
    num_qubits = len(qubits)

    # First add SWAP gates to reverse the order of the qubits:
    # for i in range(math.floor(num_qubits / 2)):
    #     qft_circuit.swap(qubits[i], qubits[-i - 1])

    # Start on the last qubit and work to the first.
    for k in reversed(range(num_qubits)):
        # Apply the controlled rotations, with weights (angles) defined by the distance to the
        # control qubit. These angles are the negative of the angle used in the QFT.
        # Start on the last qubit and iterate until the qubit after k.
        # When num_qubits==1, this loop does not run.
        for j in reversed(range(1, num_qubits - k)):
            angle = -2 * math.pi / (2 ** (j + 1))
            qft_circuit.cphaseshift(qubits[k + j], qubits[k], angle)

        # Then add a Hadamard gate
        qft_circuit.h(qubits[k])

    return qft_circuit

In [4]:
def modular_exponentiation_amod15(
    counting_qubits: QubitSetInput, aux_qubits: QubitSetInput, integer_a: int
) -> Circuit:
    """
    Construct a circuit object corresponding the modular exponentiation of a^x Mod 15

    Args:
        counting_qubits (QubitSetInput): Qubits defining the counting register
        aux_qubits (QubitSetInput) : Qubits defining the auxilary register
        integer_a (int) : Any integer that satisfies 1 < a < N and gcd(a, N) = 1.
    Returns:
        Circuit: Circuit object that implements the modular exponentiation of a^x Mod 15
    """

    # Instantiate circuit object
    mod_exp_amod15 = Circuit()

    for x in counting_qubits:
        r = 2**x
        if integer_a not in [2, 7, 8, 11, 13]:
            raise ValueError("integer 'a' must be 2,7,8,11 or 13")
        for iteration in range(r):
            if integer_a in [2, 13]:
                mod_exp_amod15.cswap(x, aux_qubits[0], aux_qubits[1])
                mod_exp_amod15.cswap(x, aux_qubits[1], aux_qubits[2])
                mod_exp_amod15.cswap(x, aux_qubits[2], aux_qubits[3])
            if integer_a in [7, 8]:
                mod_exp_amod15.cswap(x, aux_qubits[2], aux_qubits[3])
                mod_exp_amod15.cswap(x, aux_qubits[1], aux_qubits[2])
                mod_exp_amod15.cswap(x, aux_qubits[0], aux_qubits[1])
            if integer_a == 11:
                mod_exp_amod15.cswap(x, aux_qubits[1], aux_qubits[3])
                mod_exp_amod15.cswap(x, aux_qubits[0], aux_qubits[2])
            if integer_a in [7, 11, 13]:
                for q in aux_qubits:
                    mod_exp_amod15.cnot(x, q)

    return mod_exp_amod15

In [5]:
def get_factors_from_results(
    results: Dict[str, Any],
    integer_N: int,
    integer_a: int,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Function to postprocess dictionary returned by run_shors_algorithm
        and pretty print results

    Args:
        results (Dict[str, Any]): Results associated with quantum phase estimation run as produced
            by run_shors_algorithm
        integer_N (int) : The integer to be factored
        integer_a (int) : Any integer that satisfies 1 < a < N and gcd(a, N) = 1.
        verbose (bool) : If True, prints aggregate results (default is False)
    Returns:
        Dict[str, Any]: Factors of the integer N
    """

    # unpack results
    measurement_counts = results["measurement_counts"]

    # get phases
    phases_decimal = _get_phases(measurement_counts)

    r_guesses = []
    factors = []
    if verbose:
        print(f"Number of Measured phases (s/r) : {len(phases_decimal)}")
    for phase in phases_decimal:
        if verbose:
            print(f"\nFor phase {phase} :")
        r = (Fraction(phase).limit_denominator(integer_N)).denominator
        r_guesses.append(r)
        if verbose:
            print(f"Estimate for r is : {r}")
        factor = [
            math.gcd(integer_a ** (r // 2) - 1, integer_N),
            math.gcd(integer_a ** (r // 2) + 1, integer_N),
        ]
        factors.append(factor[0])
        factors.append(factor[1])
        if verbose:
            print(f"Factors are : {factor[0]} and {factor[1]}")
    factors_set = set(factors)
    factors_set.discard(1)
    factors_set.discard(integer_N)
    if verbose:
        print(f"\n\nNon-trivial factors found are : {factors_set}")

    aggregate_results = {"guessed_factors": factors_set}

    return aggregate_results


In [6]:
def _get_phases(measurement_counts: Counter) -> List[float]:
    """
    Get phase estimate from measurement_counts using top half qubits

    Args:
        measurement_counts (Counter) : measurement results from a device run
    Returns:
        List[float] : decimal phase estimates
    """

    # Aggregate the results (i.e., ignore/trace out the query register qubits):
    if not measurement_counts:
        return None

    # First get bitstrings with corresponding counts for counting qubits only (top half)
    num_counting_qubits = int(len(list(measurement_counts.keys())[0]) / 2)

    bitstrings_precision_register = [key[:num_counting_qubits] for key in measurement_counts.keys()]

    # Then keep only the unique strings
    bitstrings_precision_register_set = set(bitstrings_precision_register)
    # Cast as a list for later use
    bitstrings_precision_register_list = list(bitstrings_precision_register_set)

    # Now create a new dict to collect measurement results on the precision_qubits. Keys are given
    # by the measurement count substrings on the register qubits. Initialize the counts to zero.
    precision_results_dict = {key: 0 for key in bitstrings_precision_register_list}

    # Loop over all measurement outcomes
    for key in measurement_counts.keys():
        # Save the measurement count for this outcome
        counts = measurement_counts[key]
        # Generate the corresponding shortened key (supported only on the precision_qubits register)
        count_key = key[:num_counting_qubits]
        # Add these measurement counts to the corresponding key in our new dict
        precision_results_dict[count_key] += counts

    phases_decimal = [_binary_to_decimal(item) for item in precision_results_dict.keys()]

    return phases_decimal

In [7]:
def _binary_to_decimal(binary: str) -> float:
    """
    Helper function to convert binary string (example: '01001') to decimal

    Args:
        binary (str): value to convert to decimal fraction

    Returns:
        float: decimal value
    """

    fracDecimal = 0

    # Convert fractional part of binary to decimal equivalent
    twos = 2

    for ii in range(len(binary)):
        fracDecimal += (ord(binary[ii]) - ord("0")) / twos
        twos *= 2.0

    # return fractional part
    return fracDecimal

In [8]:
def shors_algorithm(integer_N: int, integer_a: int) -> Circuit:
    """
    Creates the circuit for Shor's algorithm.
      1) Based on integer N, calculate number of counting qubits for the first register
      2) Setup same number of auxiliary qubits for the second register
         and apply modular exponentian function
      3) Apply inverse_QFT

    Args:
        integer_N (int) : The integer N to be factored
        integer_a (int) : Any integer 'a' that satisfies 1 < a < N and gcd(a, N) = 1.

    Returns:
        Circuit: Circuit object that implements the Shor's algorithm
    """

    # validate the inputs
    if integer_N < 1 or integer_N % 2 == 0:
        raise ValueError("The input N needs to be an odd integer greater than 1.")
    if integer_a >= integer_N or math.gcd(integer_a, integer_N) != 1:
        raise ValueError('The integer "a" needs to satisfy 1 < a < N and gcd(a, N) = 1.')

    # calculate number of qubits needed
    n = int(np.ceil(np.log2(integer_N)))
    m = n

    counting_qubits = [*range(n)]
    aux_qubits = [*range(n, n + m)]

    shors_circuit = Circuit()

    # Initialize counting and aux qubits
    shors_circuit.h(counting_qubits)
    shors_circuit.x(aux_qubits[0])

    # Apply modular exponentiation
    shors_circuit.add_circuit(modular_exponentiation_amod15(counting_qubits, aux_qubits, integer_a))

    # Apply inverse QFT
    shors_circuit.add_circuit(inverse_qft_noswaps(counting_qubits))

    return shors_circuit


In [9]:
def run_shors_algorithm(circuit: Circuit, device: Device, shots: Optional[int] = 1000,) -> Dict[str, Any]:
    """
    Function to run Shor's algorithm and return measurement counts.

    Args:
        circuit (Circuit): Shor's algorithm circuit
        device (Device): Braket device backend
        shots (Optional[int]) : Number of measurement shots (default is 1000).
            0 shots results in no measurement.

    Returns:
        Dict[str, Any]: measurements and results from running Shors's algorithm
    """

    task = device.run(circuit, shots=shots)

    result = task.result()

    out = {
        "measurements": result.measurements,
        "measured_qubits": result.measured_qubits,
        "measurement_counts": result.measurement_counts,
        "measurement_probabilities": result.measurement_probabilities,
    }

    return out

### Prepare inputs for Shor's Algorithm

In [10]:
N = 15 # Integer to factor (currently 15, 21, 35 work)
a = 8 # Any integer that satisfies 1 < a < N and gcd(a, N) = 1.


shors_circuit = shors_algorithm(N, a)

### Run on a local simulator

In [11]:
local_simulator = LocalSimulator()

output = run_shors_algorithm(shors_circuit, local_simulator)

guessed_factors = get_factors_from_results(output, N, a)

Number of Measured phases (s/r) : 4

For phase 0.75 :
Estimate for r is : 4
Factors are : 3 and 5

For phase 0.5 :
Estimate for r is : 2
Factors are : 1 and 3

For phase 0.25 :
Estimate for r is : 4
Factors are : 3 and 5

For phase 0.0 :
Estimate for r is : 1
Factors are : 15 and 1


Non-trivial factors found are : {3, 5}


### Conclusion

This routine demonstartes that Shor's algorithm can be used to factor large numbers efficiently when used on a Quantum Computer.